# Binance FR trade check

This notebook:
- exports parquet data from the local API (port 19131)
- finds strategies with open trades but no hedge trades


In [ ]:
from pathlib import Path
import urllib.request
import shutil

BASE_DIR = Path(".").resolve()
OUTPUT_DIR = BASE_DIR / "output"
API_BASE = "http://localhost:19131"
CLEAN_OUTPUT = False

if CLEAN_OUTPUT and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def download(path, out_path):
    url = f"{API_BASE}/{path}"
    try:
        with urllib.request.urlopen(url, timeout=30) as resp:
            data = resp.read()
        if not data:
            print(f"{path}: empty response")
            return False
        out_path.write_bytes(data)
        size = out_path.stat().st_size
        print(f"{path}: ok ({size} bytes)")
        return True
    except Exception as exc:
        print(f"{path}: failed ({exc})")
        if out_path.exists():
            out_path.unlink()
        return False

signal_kinds = [
    "signals_arb_open",
    "signals_arb_hedge",
    "signals_arb_cancel",
    "signals_arb_close",
]

for kind in signal_kinds:
    download(f"signals/{kind}/export", OUTPUT_DIR / f"{kind}.parquet")

download("order_updates/export", OUTPUT_DIR / "order_updates.parquet")
download("trade_updates/export", OUTPUT_DIR / "trade_updates.parquet")

print("output files:")
for path in sorted(OUTPUT_DIR.glob("*.parquet")):
    print(" -", path.name)


In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = OUTPUT_DIR

def load_parquet(path):
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

def extract_strategy_id(val):
    try:
        return (int(val) >> 32) & 0xFFFFFFFF
    except Exception:
        return pd.NA

def first_non_null(series):
    for v in series:
        if pd.notna(v):
            return v
    return pd.NA

df_trade = load_parquet(DATA_DIR / "trade_updates.parquet")
df_order = load_parquet(DATA_DIR / "order_updates.parquet")
df_open_sig = load_parquet(DATA_DIR / "signals_arb_open.parquet")
df_close_sig = load_parquet(DATA_DIR / "signals_arb_close.parquet")

if not df_trade.empty:
    df_trade["strategy_id"] = df_trade["client_order_id"].apply(extract_strategy_id)
if not df_order.empty:
    df_order["strategy_id"] = df_order["client_order_id"].apply(extract_strategy_id)

sig_frames = []
if not df_open_sig.empty:
    sig_frames.append(df_open_sig)
if not df_close_sig.empty:
    sig_frames.append(df_close_sig)

if sig_frames:
    df_sig = pd.concat(sig_frames, ignore_index=True)
else:
    df_sig = pd.DataFrame()

if not df_sig.empty and {"strategy_id", "opening_venue", "hedging_venue"}.issubset(df_sig.columns):
    venue_map = (
        df_sig[["strategy_id", "opening_venue", "hedging_venue"]]
        .dropna(subset=["strategy_id"])
        .groupby("strategy_id", sort=False)
        .agg(
            opening_venue=("opening_venue", first_non_null),
            hedging_venue=("hedging_venue", first_non_null),
        )
        .reset_index()
    )
else:
    venue_map = pd.DataFrame(columns=["strategy_id", "opening_venue", "hedging_venue"])

if "trading_venue" not in df_trade.columns and "trading_venue" in df_order.columns:
    df_trade = df_trade.merge(
        df_order[["order_id", "strategy_id", "trading_venue", "symbol"]].drop_duplicates(),
        on=["order_id", "strategy_id"],
        how="left",
        suffixes=("", "_order"),
    )
elif "trading_venue" in df_trade.columns and "trading_venue" in df_order.columns:
    if df_trade["trading_venue"].isna().any():
        order_tv = (
            df_order[["order_id", "strategy_id", "trading_venue"]]
            .drop_duplicates()
            .set_index(["order_id", "strategy_id"])["trading_venue"]
        )
        def fill_tv(row):
            if pd.notna(row.get("trading_venue")):
                return row.get("trading_venue")
            return order_tv.get((row.get("order_id"), row.get("strategy_id")), pd.NA)
        df_trade["trading_venue"] = df_trade.apply(fill_tv, axis=1)

if "symbol" in df_trade.columns and "symbol" in df_order.columns:
    if df_trade["symbol"].isna().any():
        order_sym = (
            df_order[["order_id", "strategy_id", "symbol"]]
            .drop_duplicates()
            .set_index(["order_id", "strategy_id"])["symbol"]
        )
        def fill_symbol(row):
            if pd.notna(row.get("symbol")):
                return row.get("symbol")
            return order_sym.get((row.get("order_id"), row.get("strategy_id")), pd.NA)
        df_trade["symbol"] = df_trade.apply(fill_symbol, axis=1)

df_trade = df_trade.merge(venue_map, on="strategy_id", how="left")

def classify_order_side(row):
    tv = row.get("trading_venue")
    if pd.isna(tv):
        return "unknown"
    if tv == row.get("opening_venue"):
        return "open"
    if tv == row.get("hedging_venue"):
        return "hedge"
    return "unknown"

df_trade["order_side"] = df_trade.apply(classify_order_side, axis=1)

summary = (
    df_trade.groupby("strategy_id", dropna=False)
    .agg(
        open_trades=("order_side", lambda s: (s == "open").sum()),
        hedge_trades=("order_side", lambda s: (s == "hedge").sum()),
        total_trades=("order_side", "size"),
    )
    .reset_index()
)

if "symbol" in df_trade.columns:
    sym_map = (
        df_trade.groupby("strategy_id", dropna=False)["symbol"]
        .agg(first_non_null)
        .reset_index()
    )
    summary = summary.merge(sym_map, on="strategy_id", how="left")

loss_candidates = summary[(summary["open_trades"] > 0) & (summary["hedge_trades"] == 0)]
loss_candidates = loss_candidates.sort_values(["open_trades", "total_trades"], ascending=False)

print(f"loss candidates: {len(loss_candidates)} strategies")
loss_candidates.head(50)

loss_strategy_ids = loss_candidates["strategy_id"].dropna().astype("int64").tolist()
loss_strategy_ids
